In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 
from pyspark.sql.window import Window

In [0]:
%run /Users/gainrakesh2001@gmail.com/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","gross_price","Data Source")

In [0]:
 # created widgets becuase dynamically change the catalog,data_source
# suppose this can run different environment like dev ,test,prod e.g (dev_fmcg,test_fmcg,prod_fmcg) and (prod_fmcg,test_fmcg,dev_fmcg) etc
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"/Volumes/workspace/default/sportsbar_raw/{data_source}/"
print(catalog,data_source)
print(base_path)

### Bronze

In [0]:
# loading the data into in df
df = spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load(base_path)

df = df.withColumn("read_timestamp",F.current_timestamp())
df = df.select("*","_metadata.file_name","_metadata.file_size")
display(df)

In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

### Silver Processing

In [0]:
df_bronze = spark.sql(f"""select * from {catalog}.{bronze_schema}.{data_source}""")
df_bronze.show()

Transformation

Normalization month field

In [0]:
df_bronze.select('month').distinct().show()

In [0]:
date_format = ["yyyy/MM/dd","dd/MM/yyyy","yyyy-MM-dd","dd-MM-yyyy"]

df_silver = df_bronze.withColumn(
    "month" ,
    F.coalesce(
        F.try_to_date(F.col("month"), "yyyy/MM/dd"),
        F.try_to_date(F.col("month"), "dd/MM/yyyy"),
        F.try_to_date(F.col("month"), "yyyy-MM-dd"),
        F.try_to_date(F.col("month"), "dd-MM-yyyy")
    )
    
)

In [0]:
df_silver.select('month').distinct().show()

In [0]:
display(df_silver)

In [0]:
# we are validating the gross_price column, converting only valid numeric values to double , fixing negative prices by making them positive and replacing all numeric values with 0

df_silver = df_silver.withColumn(
    "gross_price" , 
    F.when(F.col("gross_price").rlike(r'^-?\d+(\.\d+)?$'),
        F.when(F.col("gross_price").cast("double") < 0 , -1 * F.col("gross_price").cast("double"))
        .otherwise(F.col("gross_price").cast("double"))
        )
    .otherwise(0)
)


In [0]:
df_silver.show()

In [0]:
# We enrich the silver dataset by performing a inner join with product table to fetch the correct product_code for each product_id.

df_products = spark.table("fmcg.silver.products")

df_silver_aliased = df_silver.alias("silver")
df_products_aliased = df_products.select("product_id","product_code").alias("prod")

df_joined = df_silver_aliased.join(df_products_aliased, F.col("prod.product_id") == F.col("silver.product_id"), "inner")
df_joined = df_joined.select(
    F.col("silver.product_id"),
    F.col("prod.product_code"),
    F.col("silver.month"),
    F.col("silver.gross_price"),
    F.col("silver.read_timestamp"),
    F.col("silver.file_name"),
    F.col("silver.file_size")
)
df_joined.show(5)




In [0]:
df_joined.write\
        .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
        .option("mergeSchema","true")\
        .mode("overwrite")\
        .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

### Gold processing

In [0]:
df_silver = spark.sql(f"""
                      select * from {catalog}.{silver_schema}.{data_source} 
                      """)
df_silver.show()

In [0]:
df_gold = df_silver.select("product_code","month","gross_price").show(5)

In [0]:
df_gold = df_silver.select("product_code","month","gross_price")
df_gold.write\
                .format("delta")\
                .option("delta.enableChangeDataFeed", "true")\
                .option("mergeSchema","true")\
                .mode("overwrite")\
                .saveAsTable(f"{catalog}.{gold_schema}.sb_dm_{data_source}") # this gold data for sports bar company that's why sb_dm


### Merging Data source with parent

In [0]:
df_gold_price = spark.table("fmcg.gold.sb_dm_gross_price")
display(df_gold_price)

In [0]:
df_gold_price = (
    df_gold_price.withColumn("year", F.year("month"))

    # 0 = non-zero price , 1 = zero price 
    .withColumn("is_zero" , F.when(F.col("gross_price") != 0 , 0).otherwise(1)) # collecting only non zero price
)

w = (
    Window.partitionBy("product_code" , "year")
    .orderBy(F.col("is_zero").asc(), F.col("month").desc())
)
df_gold_latest_price = (
    df_gold_price.withColumn("rnk", F.row_number().over(w))
                .filter(F.col("rnk") == 1)
)



In [0]:
df_gold_latest_price.show()

In [0]:
df_gold_latest_price = df_gold_latest_price.withColumnRenamed("gross_price","price_inr")

In [0]:
df_gold_latest_price.show(5)

In [0]:
df_gold_latest_price.select("product_code","price_inr","year")

df_gold_latest_price = df_gold_latest_price.withColumn("year" , F.col("year").cast("string"))


In [0]:
df_gold_latest_price.printSchema()

In [0]:
# Upsert the data into the delta table
df_gold_latest_price = df_gold_latest_price.select("product_code","price_inr","year") 
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_gross_price")

delta_table.alias("target").merge(
    source= df_gold_latest_price.alias("source"),
    condition = "target.product_code = source.product_code"
).whenMatchedUpdate(
    set = {
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "price_inr": "source.price_inr",
        "year": "source.year"
    }
).execute()